In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_62_try_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 63 ###

def grab_subset_of_data_75(original_df, question_of_interest):
    # GPU-only: boolean mask of columns containing the question
    col_mask = original_df.columns.str.contains(question_of_interest)
    # GPU: select columns by label list instead of .loc
    selected_cols = original_df.columns[col_mask]
    sub_df = original_df[selected_cols]

    # GPU: rename columns by stripping everything up to "-"
    new_cols = sub_df.columns.str.replace(r'^.*-\s*', '', regex=True)
    # use set_axis to avoid Python‐level list comprehension
    sub_df = sub_df.set_axis(new_cols, axis=1)

    # GPU: drop rows where all values are null via boolean mask
    row_mask = ~sub_df.isnull().all(axis=1)
    return sub_df[row_mask]